# Indexing notebook — Assignment 3

**Goal:** build a reproducible, citeable retrieval index of Python 3.13 reference documentation. Official text was extracted from the public, PSF-licensed CPython `pydoc_data.topics` snapshot into eight source-document groups in `source_docs/`. Every section has a `topic:` locator.

**Design:** preserve section/code layout, split by paragraph into ~390-word chunks with 45-word overlap, save stable SHA-256 IDs, and commit text plus Google Gemini Embedding 2 vectors to SQLite (`index/corpus.sqlite3`). SQLite is a persistent vector store (float32 blobs, cosine similarity and BM25) analogous to a simple vector DB. The build resumes and embeds **only missing** vectors. Query embedding is never done during indexing.

This notebook and `server.py` reuse the **same** functions from `rag_core.py` so add-document ingestion cannot diverge from initial indexing.

In [ ]:
from pathlib import Path
from collections import Counter
from rag_core import (ROOT, SOURCES_DIR, manifest_map, clean_text, chunk_file,
                      build_index, index_summary, retrieve, available_vectors)
manifest = manifest_map()
print(f"Documents: {len(manifest)}; source words: {sum(m['words'] for m in manifest.values()):,}")
for name, doc in sorted(manifest.items()):
    print(f"  {name:25} {doc['words']:>6,} words | {doc['url']}")

## Extraction quality and metadata

`rag_core.read_document()` handles Markdown/text with source-defined topic boundaries, HTML using BeautifulSoup after stripping navigation/scripts, and PDF by extracted page. `clean_text()` normalizes line endings and fixes soft hyphenation but preserves whitespace significant to Python code. For example, each chunk has `{id, source, location, url, filepath, text, n_words}`.

In [ ]:
example = chunk_file(SOURCES_DIR / "simple_statements.md")
print("Sample chunks:", len(example))
for ch in example[:2]:
    print({key: ch[key] for key in ('id','source','location','url','n_words')})
    print(ch['text'][:200].replace('\n', ' '), "...\n")

## Build / resume the persisted index

With `GEMINI_API_KEY` (or `.env`), Gemini Embedding 2 is called once for each missing document vector and committed immediately. Without a key, the source chunks are **saved but remain pending**, with an explicit warning; no vectors are fabricated. The ready-to-commit SQLite index also supports offline BM25 evaluation. Use `uv run python main.py index` to run this notebook as a script.

In [ ]:
result = build_index()
print('Index summary:')
for key in ('documents','source_words','chunks','saved_gemini_vectors','warning','embedding'):
    if key in result:
        print(f'  {key}: {result[key]}')

## Spot-check citations and vector persistence

`index_summary()` reads disk and performs no embedding. `retrieve()` supports ordinary BM25, structure-aware BM25 and Gemini-vector/BM25 reciprocal-rank fusion. Hybrid search is only labeled “hybrid” when *all* chunk vectors exist; otherwise the actual fallback is explicitly named.

In [ ]:
print(index_summary())
hits = retrieve('What is Python nonlocal and its enclosing scope?', k=3, design='hybrid')
for h in hits:
    print(f"[CITE:{h['id']}] {h['source']} :: {h['location']} ({h['design']})")